# Pipeline v14 (v15 hybrid) — CoT LoRA Fine-tune + Gold-FOL prompt + optillm re2
**EXACT 2026 — XAI Challenge @ IJCNN | Qwen3-8B + Unsloth QLoRA | Kaggle T4×2**

## Vì sao v14 tồn tại (và vì sao nó là đòn bẩy duy nhất còn lại)

Bằng chứng từ v13.3:
- Z3 bucket đã đụng trần ngữ nghĩa ~60% — không vượt được bằng entailment cổ điển
- Aggregator bất kỳ trên signals hiện tại chỉ chạm 53-54%
- **Đòn bẩy duy nhất: nâng chất lượng Qwen-CoT** — base model chỉ 33%, quá thấp

Dataset có `explanation` field — 630 reasoning chains gold-supervised (v4 cleaned, 318 records).
Đây là *chính xác* pedagogical reasoning mà nhãn dataset thưởng. **Chưa khai thác đến v14.**

## v15 Hybrid: 2 cải tiến mới lai từ Pipeline B + optillm

### Cải tiến 1: Gold FOL trong CoT prompt (từ Pipeline B / exact_70)

| | v14 cũ (NL-only) | v15 hybrid (NL + FOL) |
|---|---|---|
| Prompt | `P1. All students enroll` | `P1. [NL]  All students enroll` <br> `      [FOL] ∀x (Enroll(x))` |
| Lý do | Chưa khai thác gold FOL | Pipeline B đạt **70%** nhờ model thấy cả formal notation |
| Model học gì | Chỉ reasoning từ NL | Reasoning từ NL + kiểm chứng bằng FOL structure |
| Rủi ro | Không | PHẢI match train↔inference (bài học -11pp từ v13.4) |

**Vì sao gold FOL quan trọng:** dataset có `premises-FOL` field chứa annotation chuẩn.
Pipeline B (exact_70) đạt 70.1% trên 411 records — không dùng Z3 gì cả, chỉ nhờ model ĐỌC
cả NL lẫn FOL trong prompt. v14 cũ bỏ qua field này hoàn toàn. v15 khắc phục.

### Cải tiến 2: optillm `re2` (ReRead) — kỹ thuật test-time compute rẻ nhất

Nguồn: [optillm](https://github.com/algorithmicsuperintelligence/optillm) — bộ 20+ kỹ thuật
tối ưu inference cho LLM. Trong số đó:
- MCTS, MoA, CePO: quá nặng cho T4 12h
- self_consistency: v13.6 đã có (BoN-SC)
- **re2 (ReRead)**: cực rẻ, proven boost reasoning — chỉ cần lặp lại câu hỏi cuối prompt

Paper gốc: [Re-Reading Improves Reasoning in LLMs](https://arxiv.org/abs/2309.06275)

Cách hoạt động: thêm `"Read the question again carefully: {question}"` vào cuối user message.
Model "đọc lại" → attention pattern tập trung hơn vào question → reasoning tốt hơn.
Chi phí thêm: ~20 tokens/câu (≈ 0 trên tổng thể).

## Data prep
- 318 samples v4 (cleaned) × ~2 questions/sample = ~630 cặp `(premises, question, explanation, gold)`
- Train/val split sample-level: **SEED=42, 80/20** — KHỚP HOÀN TOÀN với v13.x
- Training target = `explanation + "\n\nFinal Answer: <gold>"` (append nếu chưa có marker)
- **MỚI (v15):** user message giờ chứa cả `[NL]` + `[FOL]` + re2 ReRead

## Hyperparams
- Unsloth QLoRA 4-bit | r=16, α=16, dropout=0
- LR 1e-4, warmup 10, weight_decay 0.01
- 2 epochs × ~254 examples/epoch (80% of 318)
- Batch 2 × grad_accum 4 = effective batch 8
- max_seq_length 4096 (prompt dài hơn ~15% do thêm FOL lines)

## Mục tiêu định lượng
- Qwen-CoT val acc: **33% → ≥ 50%** (success threshold, giống v14 cũ)
- Kỳ vọng thực tế: **55-65%** (exact_70 đạt 70% trên 411 raw, ta trên 318 clean → khó hơn nhưng prompt tốt hơn)
- Nếu đạt ≥55%: cắm LoRA vào v13.6 pipeline → kỳ vọng tổng 80%+ (vượt 79.8% hiện tại)

## Inference compatibility
- Train với `enable_thinking=False` — reasoning trực tiếp trong assistant text
- Lưu LoRA adapter format PEFT → vLLM nạp được qua `enable_lora=True`
- Inline eval cuối notebook dùng Unsloth `FastLanguageModel.for_inference`
- **CRITICAL:** `SYSTEM_PROMPT` trong notebook này PHẢI byte-identical với `V14_COT_SYSTEM`
  trong v13.5/v13.6. Đã verify bằng script (xem apply_hybrid.py)

## Output
- `/kaggle/working/qwen3_cot_lora_v14_v4/` — PEFT adapter (dùng cho v13.5/v13.6 sau)
- `/kaggle/working/v14_eval_results.json` — val accuracy + per-question results

##  Thứ tự chạy (quan trọng)
1. **Chạy notebook này TRƯỚC** → train LoRA với prompt mới (NL+FOL+re2)
2. Upload LoRA adapter lên Kaggle Dataset
3. Mới chạy v13.5/v13.6 (chúng load LoRA v14 + dùng prompt khớp)

Nếu chạy v13.x với LoRA v14 **cũ** (train bằng prompt NL-only) → mismatch → accuracy tụt.
Đây chính là bài học -11pp từ v13.4.


In [1]:
#!/usr/bin/env python3
"""
notebook_v14_cot_lora_finetune.py
EXACT 2026 -- v14: Train Qwen3-8B LoRA on dataset's `explanation` field
to produce pedagogical CoT reasoning + Final Answer marker.
Output: /kaggle/working/qwen3_cot_lora_v14 (PEFT adapter for vLLM).
"""


"\nnotebook_v14_cot_lora_finetune.py\nEXACT 2026 -- v14: Train Qwen3-8B LoRA on dataset's `explanation` field\nto produce pedagogical CoT reasoning + Final Answer marker.\nOutput: /kaggle/working/qwen3_cot_lora_v14 (PEFT adapter for vLLM).\n"

In [2]:
# ================= Kaggle T4 fix =================
import os, shutil, glob
STUB_DIR = "/tmp/cuda_stubs"; os.makedirs(STUB_DIR, exist_ok=True)
stub = os.path.join(STUB_DIR, "libcuda.so")
if os.path.exists(stub) or os.path.islink(stub): os.remove(stub)
for c in ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so",
          *glob.glob("/usr/**/libcuda.so*", recursive=True)]:
    if os.path.exists(c) and not os.path.islink(c):
        os.symlink(c, stub); break
os.environ["LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LD_LIBRARY_PATH", "")
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
print("Kaggle T4 fixes applied")


Kaggle T4 fixes applied


In [3]:
# ================= INSTALL =================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages",
                "unsloth", "trl<=0.24.0", "datasets<4.4.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--break-system-packages",
                "--no-cache-dir", "peft", "accelerate", "bitsandbytes"], check=True)
import torch
print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 27.4 MB/s eta 0:00:00
PyTorch 2.10.0+cu128 CUDA True


In [4]:
# ================= CONFIG =================
MODEL_PATH       = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
# ---- Robust Kaggle path resolver (handles mount-path variations) ----
import os as _os
def _resolve(cands, label="path"):
    for p in cands:
        if _os.path.exists(p):
            print(f"  resolved {label}: {p}")
            return p
    print(f"  WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

TRAIN_PATH = _resolve([
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
], "TRAIN")
TEST_PATH = _resolve([
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")
DATASET_PATH = TRAIN_PATH    # eval notebooks use the labeled train file

# DATASET_PATH set by resolver below
LORA_OUTPUT_DIR  = "/kaggle/working/qwen3_cot_lora_v14_v4"  # NEW: trained on v4 data
CHECKPOINT_DIR   = "/kaggle/working/cot_lora_ckpt"
EVAL_OUTPUT_PATH = "/kaggle/working/v14_eval_results.json"

# Split (MUST match v13: CAL_SEED=42, CAL_TRAIN_RATIO=0.80, sample-level)
SEED          = 42
TRAIN_RATIO   = 0.80

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT   = True

LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

EPOCHS         = 2
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LEARNING_RATE  = 1e-4
WARMUP_STEPS   = 10
WEIGHT_DECAY   = 0.01
OPTIMIZER      = "adamw_8bit"
LR_SCHEDULER   = "linear"
LOGGING_STEPS  = 10

# Must match inference (we use plain CoT, not thinking-mode tokens)
ENABLE_THINKING_TRAIN = False

# Inline eval
EVAL_MAX_NEW_TOKENS = 512
EVAL_TEMPERATURE    = 0.1   # low temp for evaluation
EVAL_N_SAMPLES      = None  # None = all val; set e.g. 30 for quick sanity

# System prompt -- must be identical at training and inference
# v15: FOL-aware system prompt (hybridized from Pipeline B / exact_70).
# CRITICAL: must be byte-identical to V14_COT_SYSTEM in v13.5/v13.6 inference.
SYSTEM_PROMPT = (
    "You are a careful logic tutor specializing in First-Order Logic (FOL). "
    "Given premises (each shown in natural language and FOL notation) and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'), "
    "using the FOL notation to check entailment precisely. "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)
# v15 hybridization flags (Pipeline B prompt + optillm re2)
INCLUDE_FOL_IN_PROMPT = True   # Pipeline B: show [NL]+[FOL] in prompt (key 70% driver)
USE_RE2_REREAD        = True   # optillm re2 (ReRead): re-read question, cheap reasoning boost
print("Config loaded")


  resolved TRAIN: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json
  resolved TEST: /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
Config loaded


In [5]:
# ================= LOAD MODEL + ATTACH LoRA =================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trn:,}/{tot:,} ({100*trn/tot:.2f}%)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable: 43,646,976/4,761,498,624 (0.92%)


In [6]:
# ================= BUILD CoT DATASET =================
import json, random, re
from datasets import Dataset

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = json.load(f)
n = len(raw)
print(f"Dataset samples: {n}")

# Sample-level split, deterministic, MATCHES v13.3
rng = random.Random(SEED)
ids = list(range(n)); rng.shuffle(ids)
n_tr = int(n * TRAIN_RATIO)
train_ids = set(ids[:n_tr]); val_ids = set(ids[n_tr:])
train_samples = [raw[i] for i in range(n) if i in train_ids]
val_samples   = [raw[i] for i in range(n) if i in val_ids]
print(f"Train: {len(train_samples)} samples | Val: {len(val_samples)} samples")

def build_user_msg(premises_nl, question, premises_fol=None):
    # v15: hybridized prompt -- NL + gold FOL (Pipeline B) + re2 ReRead (optillm)
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. [NL]  {p}")
        if INCLUDE_FOL_IN_PROMPT and premises_fol and i < len(premises_fol) and premises_fol[i]:
            lines.append(f"      [FOL] {premises_fol[i]}")
    body = "\n".join(lines) + f"\n\n### Question\n{question}"
    if USE_RE2_REREAD:
        body += f"\n\nRead the question again carefully: {question}"
    return body

def build_assistant_msg(explanation, gold):
    # Append "Final Answer: X" unless explanation already ends with such a line
    txt = explanation.strip()
    last = txt.split("\n")[-1].lower()
    if not (re.search(r"\bfinal\s+answer\b", last) or
            re.search(r"\banswer\s*:", last)):
        txt = f"{txt}\n\nFinal Answer: {gold}"
    return txt

# Build records (one per question)
def to_records(samples):
    out = []
    for s in samples:
        nls = s.get("premises-NL", [])
        fols = s.get("premises-FOL", [])
        for q, a, e in zip(s.get("questions", []), s.get("answers", []),
                            s.get("explanation", [])):
            if not e or not str(a).strip():
                continue
            out.append({"conversations": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": build_user_msg(nls, q, fols)},
                {"role": "assistant", "content": build_assistant_msg(e, a)},
            ]})
    return out

train_records = to_records(train_samples)
print(f"Train examples (questions): {len(train_records)}")
print("\n--- Sample training record ---")
print("USER:", train_records[0]["conversations"][1]["content"][:300], "...")
print("\nASSISTANT:", train_records[0]["conversations"][2]["content"][:400])

train_ds = Dataset.from_list(train_records)
print(f"\nDataset: {train_ds}")


Dataset samples: 318
Train: 254 samples | Val: 64 samples
Train examples (questions): 504

--- Sample training record ---
USER: ### Premises
P1. [NL]  All students must enroll in at least one core subject.
      [FOL] ∀x (EnrollsCoreSubject(x))
P2. [NL]  If a student attends all lectures, they will have a higher chance of passing the course.
      [FOL] ∀x (AttendsLectures(x) → HigherChancePass(x))
P3. [NL]  If a student doe ...

ASSISTANT: From premise 6 (There exists at least one student who participates in an academic competition), premise 8 (If a student joins the research group, they must contribute to at least one published paper), premise 12 (If joining the research group requires contributing to a published paper, then at least one student must be involved in research), and premise 16 (If a student joins the research group, t

Dataset: Dataset({
    features: ['conversations'],
    num_rows: 504
})


In [7]:
# ================= TRAINER (responses-only) =================
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def formatting_func(examples):
    convos = examples["conversations"]
    def _fmt(c):
        try:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False,
                enable_thinking=ENABLE_THINKING_TRAIN)
        except TypeError:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False)
    if len(convos) > 0 and isinstance(convos[0], dict):
        return [_fmt(convos)]
    return [_fmt(c) for c in convos]

sft = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIMIZER,
    lr_scheduler_type=LR_SCHEDULER,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)
trainer_obj = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds,
                         args=sft, formatting_func=formatting_func)
trainer_obj = train_on_responses_only(
    trainer_obj,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print(f"Trainer ready | examples={len(train_ds)} | effective batch={BATCH_SIZE*GRAD_ACCUM}")


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/504 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/504 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/504 [00:00<?, ? examples/s]

Trainer ready | examples=504 | effective batch=8


In [8]:
# ================= TRAIN + SAVE LoRA =================
import time, os
t0 = time.time()
result = trainer_obj.train()
dur_min = (time.time() - t0) / 60
print(f"Training done in {dur_min:.1f} min | final loss = {result.training_loss:.4f}")

# Save adapter
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"Saved LoRA -> {LORA_OUTPUT_DIR}")
print("Contents:", os.listdir(LORA_OUTPUT_DIR))


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 504 | Num Epochs = 2 | Total steps = 126
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.315415
20,0.614311
30,0.562628
40,0.533200
50,0.457031
60,0.421466
70,0.391195
80,0.293909
90,0.326057
100,0.313870


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cot_lora_ckpt/checkpoint-63/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cot_lora_ckpt/checkpoint-126/tokenizer_config.json.


Training done in 49.5 min | final loss = 0.4752


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3_cot_lora_v14_v4/tokenizer_config.json.


Saved LoRA -> /kaggle/working/qwen3_cot_lora_v14_v4
Contents: ['tokenizer.json', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'README.md', 'tokenizer_config.json']


In [9]:
# ================= INLINE EVAL on VAL (vs pure_qwen 33% baseline) =================
import re, json, time
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
print("Model in inference mode")

def extract_final_answer(text):
    if not text: return "UNPARSEABLE"
    # Tier 1: 'Final Answer: X' (case-insensitive)
    m = re.search(r"final\s*answer\s*[:\-]?\s*([A-Da-d]\b|Yes|No|Unknown|YES|NO|UNKNOWN|yes|no|unknown)",
                  text, re.I)
    if m:
        a = m.group(1).strip()
        return a.upper() if a.lower() in ("yes","no","unknown") else a.upper()
    # Tier 2: standalone last line
    for line in reversed(text.strip().split("\n")):
        s = line.strip()
        if s in ("Yes","No","Unknown","A","B","C","D"): return s.upper() if len(s)>1 else s
    # Tier 3: any answer-like token in last 100 chars
    tail = text[-200:]
    for tok in ["Yes","No","Unknown","A","B","C","D"]:
        if re.search(rf"\b{tok}\b", tail):
            return tok.upper() if len(tok)>1 else tok
    return "UNPARSEABLE"

# Build val records (use same code paths as training)
val_records = []
for s in val_samples:
    nls = s.get("premises-NL", [])
    for q_idx, (q, a) in enumerate(zip(s.get("questions", []), s.get("answers", []))):
        val_records.append({"sample_id": s.get("idx", [[]])[0] if False else None,
                            "q_idx": q_idx, "user": build_user_msg(nls, q, s.get("premises-FOL", [])),
                            "gold": str(a).strip()})

if EVAL_N_SAMPLES is not None:
    val_records = val_records[:EVAL_N_SAMPLES]
print(f"Evaluating {len(val_records)} val questions...")

def gen_one(user_msg):
    msgs = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":user_msg}]
    try:
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                            return_tensors="pt", enable_thinking=ENABLE_THINKING_TRAIN).to("cuda")
    except TypeError:
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                            return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=EVAL_MAX_NEW_TOKENS,
                         temperature=EVAL_TEMPERATURE, do_sample=(EVAL_TEMPERATURE>0),
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

t0 = time.time(); results = []
correct = 0
for j, rec in enumerate(val_records):
    raw = gen_one(rec["user"])
    pred = extract_final_answer(raw)
    ok = (str(pred).upper() == str(rec["gold"]).upper())
    if ok: correct += 1
    results.append({"q_idx": rec["q_idx"], "gold": rec["gold"], "pred": pred,
                    "correct": ok, "raw_tail": raw[-300:]})
    if (j+1) % 25 == 0:
        print(f"  [{j+1}/{len(val_records)}] running acc = {correct/(j+1):.1%}  ({(time.time()-t0)/60:.1f} min)")

acc = correct / max(len(val_records), 1)
dur = (time.time() - t0) / 60
print("\n" + "="*70)
print(f"  v14 INLINE EVAL on VAL (matches v13.3 val split)")
print("="*70)
print(f"  Questions evaluated : {len(val_records)}")
print(f"  Correct             : {correct}")
print(f"  ACCURACY            : {acc:.1%}")
print(f"  Duration            : {dur:.1f} min")
print("-"*70)
print(f"  Baseline pure_qwen (v13.3 val) : 33.1%")
print(f"  Improvement                    : {acc - 0.331:+.1%}")
print(f"  Success threshold (>=50%)      : {'PASS' if acc >= 0.50 else 'FAIL'}")
print("="*70)

# Save
summary = {
    "meta": {"version": "v14_cot_lora", "seed": SEED, "train_ratio": TRAIN_RATIO,
             "lora_dir": LORA_OUTPUT_DIR, "duration_min": dur,
             "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")},
    "metrics": {"val_accuracy": acc, "correct": correct, "total": len(val_records),
                "baseline_pure_qwen": 0.331, "improvement_pp": acc - 0.331},
    "per_question": results,
}
with open(EVAL_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"\nSaved eval: {EVAL_OUTPUT_PATH}")

print("\nNEXT STEPS:")
print(f"  - LoRA adapter saved at: {LORA_OUTPUT_DIR}")
print( "  - To integrate in v13.3: in that notebook's config cell set")
print(f"      FORMALIZER_LORA_PATH = '{LORA_OUTPUT_DIR}'  (or wire a new flag)")
print( "    and pass use_lora=True to batch_generate for the CoT step.")
print( "  - Re-run v13.3 -> expect new oracle ceiling + better pure_qwen + better aggregators")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model in inference mode
Evaluating 126 val questions...


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

  [25/126] running acc = 76.0%  (2.9 min)


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  [75/126] running acc = 81.3%  (9.1 min)


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [100/126] running acc = 82.0%  (12.2 min)


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  [125/126] running acc = 80.8%  (15.7 min)

  v14 INLINE EVAL on VAL (matches v13.3 val split)
  Questions evaluated : 126
  Correct             : 102
  ACCURACY            : 81.0%
  Duration            : 15.7 min
----------------------------------------------------------------------
  Baseline pure_qwen (v13.3 val) : 33.1%
  Improvement                    : +47.9%
  Success threshold (>=50%)      : PASS

Saved eval: /kaggle/working/v14_eval_results.json

NEXT STEPS:
  - LoRA adapter saved at: /kaggle/working/qwen3_cot_lora_v14_v4
  - To integrate in v13.3: in that notebook's config cell set
      FORMALIZER_LORA_PATH = '/kaggle/working/qwen3_cot_lora_v14_v4'  (or wire a new flag)
    and pass use_lora=True to batch_generate for the CoT step.
  - Re-run v13.3 -> expect new oracle ceiling + better pure_qwen + better aggregators


## v15 Diagnostics

### Trực quan hóa dữ liệu (internal + external)
- **Internal (train v4):** 318 records, ~630 questions, có nhãn → in label distribution, premise/question count, gold-FOL fill rate
- **External (generated_v4style_300):** 300 records, ~600 questions, KHÔNG có nhãn → in structure + OOD check
- **OOD check:** so sánh predicate vocabulary giữa train và test. Bootstrap ghi: 91% predicate trong test KHÔNG có trong train → đây là OOD test thật

### Kiểm tra Overfit / Underfit
Eval LoRA trên một subset TRAIN (60 câu random) và so với VAL accuracy:
- `train_probe >> val` → **OVERFIT** (model nhớ, không tổng quát)
- Cả hai thấp (~majority baseline) → **UNDERFIT** (LoRA chưa học đủ)
- `train_probe ≈ val`, cả hai > baseline → **HEALTHY** (tổng quát hóa tốt)


In [10]:
# ============================================================
#  v15 DIAGNOSTICS -- Data Visualization (internal + external)
# ============================================================
import json as _json, re as _re
from collections import Counter as _Counter

def _v15_load(p):
    with open(p, encoding="utf-8") as f:
        return _json.load(f)

def _v15_stats(data, name, labeled=True):
    n_rec = len(data)
    n_q = sum(len(d.get("questions", [])) for d in data)
    prem = [len(d.get("premises-NL", [])) for d in data] or [0]
    qc   = [len(d.get("questions", [])) for d in data] or [0]
    print(f"\n=== {name} ===")
    print(f"  Records       : {n_rec}")
    print(f"  Questions     : {n_q}")
    print(f"  Premises/rec  : min={min(prem)} max={max(prem)} avg={sum(prem)/max(n_rec,1):.1f}")
    print(f"  Questions/rec : min={min(qc)} max={max(qc)} avg={n_q/max(n_rec,1):.1f}")
    labels = None
    if labeled:
        labels = _Counter(str(a) for d in data for a in d.get("answers", []))
        tot = sum(labels.values()) or 1
        print(f"  Label dist    :")
        for lab, c in sorted(labels.items()):
            bar = "#" * int(c / tot * 40)
            print(f"    {lab:8s}: {c:4d} ({c/tot*100:5.1f}%) {bar}")
    return labels

def _v15_pred_vocab(data):
    preds = set()
    for d in data:
        for fol in d.get("premises-FOL", []):
            for m in _re.findall(r'([A-Za-z_]\w*)\s*\(', fol or ""):
                preds.add(m.lower())
    return preds

def _v15_fol_coverage(data):
    total = sum(len(d.get("premises-FOL", [])) for d in data)
    nonempty = sum(1 for d in data for f in d.get("premises-FOL", []) if (f or "").strip())
    return nonempty, total

print("=" * 70)
print("  v15 DATA DIAGNOSTICS")
print("=" * 70)

_internal = _v15_load(TRAIN_PATH)
_v15_stats(_internal, "INTERNAL (train v4, labeled)", labeled=True)
_ne, _tot = _v15_fol_coverage(_internal)
print(f"  Gold-FOL fill : {_ne}/{_tot} premises have FOL ({_ne/max(_tot,1)*100:.1f}%)")

_has_ext = False
try:
    _external = _v15_load(TEST_PATH)
    _has_ext = True
except Exception as e:
    _external = []
    print(f"\n[!] External test not loaded: {e}")

if _has_ext:
    _v15_stats(_external, "EXTERNAL (generated_v4style, unlabeled)", labeled=False)
    _iv = _v15_pred_vocab(_internal)
    _ev = _v15_pred_vocab(_external)
    _ovl = len(_iv & _ev) / max(len(_ev), 1)
    print(f"\n=== OOD CHECK (predicate vocabulary) ===")
    print(f"  Internal predicates  : {len(_iv)}")
    print(f"  External predicates  : {len(_ev)}")
    print(f"  Overlap (ext seen)   : {_ovl*100:.1f}%")
    _verdict = "HIGH OOD" if _ovl < 0.3 else ("MODERATE OOD" if _ovl < 0.7 else "LOW OOD")
    print(f"  Verdict              : {_verdict}")
    print(f"  => External test is {'genuinely out-of-distribution (predicate names differ)' if _ovl < 0.3 else 'partially novel'};")
    print(f"     a train/val gap here is EXPECTED, not necessarily overfit.")


  v15 DATA DIAGNOSTICS

=== INTERNAL (train v4, labeled) ===
  Records       : 318
  Questions     : 630
  Premises/rec  : min=3 max=36 avg=10.7
  Questions/rec : min=1 max=2 avg=2.0
  Label dist    :
    A       :  129 ( 20.5%) ########
    B       :   41 (  6.5%) ##
    C       :   45 (  7.1%) ##
    D       :   20 (  3.2%) #
    No      :   28 (  4.4%) #
    Unknown :   90 ( 14.3%) #####
    Yes     :  277 ( 44.0%) #################
  Gold-FOL fill : 3398/3398 premises have FOL (100.0%)

=== EXTERNAL (generated_v4style, unlabeled) ===
  Records       : 300
  Questions     : 600
  Premises/rec  : min=7 max=11 avg=9.0
  Questions/rec : min=2 max=2 avg=2.0

=== OOD CHECK (predicate vocabulary) ===
  Internal predicates  : 1242
  External predicates  : 116
  Overlap (ext seen)   : 9.5%
  Verdict              : HIGH OOD
  => External test is genuinely out-of-distribution (predicate names differ);
     a train/val gap here is EXPECTED, not necessarily overfit.


In [11]:
# ============================================================
#  v15 OVERFIT / UNDERFIT CHECK (v14 LoRA)
# ============================================================
# Eval the trained LoRA on a TRAIN subset and compare to VAL accuracy.
#   - train_acc >> val_acc  -> OVERFIT
#   - both low (~ majority baseline) -> UNDERFIT
#   - train_acc ~ val_acc, both high -> healthy generalization
import random as _r

N_TRAIN_PROBE = 60   # questions sampled from TRAIN split for the probe

# Build TRAIN probe records (same code paths as eval)
_train_probe = []
for s in train_samples:
    nls = s.get("premises-NL", [])
    fols = s.get("premises-FOL", [])
    for q_idx, (q, a) in enumerate(zip(s.get("questions", []), s.get("answers", []))):
        _train_probe.append({"user": build_user_msg(nls, q, fols), "gold": str(a).strip()})
_r.Random(SEED).shuffle(_train_probe)
_train_probe = _train_probe[:N_TRAIN_PROBE]
print(f"Probing {len(_train_probe)} TRAIN questions for overfit check...")

_tc = 0
for rec in _train_probe:
    raw = gen_one(rec["user"])
    pred = extract_final_answer(raw)
    if str(pred).upper() == str(rec["gold"]).upper():
        _tc += 1
train_probe_acc = _tc / max(len(_train_probe), 1)

# val accuracy `acc` computed in the previous eval cell
val_acc = acc
gap = train_probe_acc - val_acc

# majority-class baseline on val
from collections import Counter as _C
_valgold = _C(r["gold"] for r in val_records)
_maj = max(_valgold.values()) / max(sum(_valgold.values()), 1)

print("\n" + "=" * 70)
print("  v15 OVERFIT / UNDERFIT DIAGNOSIS")
print("=" * 70)
print(f"  TRAIN-probe accuracy : {train_probe_acc:.1%}  (n={len(_train_probe)})")
print(f"  VAL accuracy         : {val_acc:.1%}  (n={len(val_records)})")
print(f"  Majority baseline    : {_maj:.1%}")
print(f"  Train-Val gap        : {gap:+.1%}")
print("-" * 70)
if val_acc < _maj + 0.05:
    print("  VERDICT: UNDERFIT -- val barely beats majority baseline.")
    print("           Consider: more epochs, higher LoRA rank, check prompt match.")
elif gap > 0.15:
    print("  VERDICT: OVERFIT -- train >> val. Model memorizes training data.")
    print("           Consider: fewer epochs, higher dropout, more data, lower rank.")
else:
    print("  VERDICT: HEALTHY -- train and val close, both above baseline.")
print("=" * 70)


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Probing 60 TRAIN questions for overfit check...


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


  v15 OVERFIT / UNDERFIT DIAGNOSIS
  TRAIN-probe accuracy : 78.3%  (n=60)
  VAL accuracy         : 81.0%  (n=126)
  Majority baseline    : 44.4%
  Train-Val gap        : -2.6%
----------------------------------------------------------------------
  VERDICT: HEALTHY -- train and val close, both above baseline.
